# GFS clamp: stato G/T poi looming 25 Hz

Fork Shiu v783. 0-200 ms solo overlay DH44/CRZ/Hugin. 200-1000 ms + LC4/LPLC2 a 25 Hz. Readout DNp01. Latenza = primo spike dopo 200 ms. Scalare OFF.


In [1]:
import os
from pathlib import Path
if not os.path.exists("Drosophila_brain_model/model.py"):
    !git clone --depth 1 https://github.com/philshiu/Drosophila_brain_model.git
    !pip install -q brian2 pandas pyarrow
%cd Drosophila_brain_model
os.makedirs("results/gfs_phased", exist_ok=True)
print("783", os.path.exists("Completeness_783.csv"), os.path.exists("Connectivity_783.parquet"))


Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 23 (delta 1), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 181.74 MiB | 34.60 MiB/s, done.
Resolving deltas: 100% (1/1), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.3 MB/s eta 0:00:00
/content/Drosophila_brain_model
783 True True


In [2]:
from time import time
import numpy as np, pandas as pd
from brian2 import Hz, ms, Network, prefs
from model import create_model, poi, silence, get_spk_trn, construct_dataframe, default_params as params
import utils as utl
prefs.codegen.target = "numpy"
config = {"path_res": "./results/gfs_phased", "path_comp": "./Completeness_783.csv", "path_con": "./Connectivity_783.parquet"}
params["n_run"] = 1
params["r_poi"] = 25 * Hz
params["t_pre"] = 200 * ms
params["t_stim"] = 800 * ms
os.makedirs(config["path_res"], exist_ok=True)
_comp = set(pd.read_csv(config["path_comp"]).iloc[:,0].astype("int64"))
def keep(name, lst):
    ok = [int(i) for i in lst if int(i) in _comp]
    print(name, "tenuti", len(ok), "/", len(lst))
    return ok


In [3]:
DH44 = keep("DH44", [720575940616153371,720575940618385718,720575940626619398,720575940622872870,720575940623011845,720575940635861146])
CRZ = keep("CRZ", [720575940609268316,720575940610895474,720575940614254534,720575940619845552,720575940620332141,720575940642666440])
HUGIN = keep("HUGIN", [720575940637851069,720575940638394446,720575940627293201,720575940643847022])
GF_L, GF_R = 720575940622838154, 720575940632499757
ann = pd.read_csv("https://raw.githubusercontent.com/flyconnectome/flywire_annotations/main/supplemental_files/Supplemental_file1_neuron_annotations.tsv", sep="\t", usecols=["root_id","cell_type"])
LC4 = keep("LC4", ann.loc[ann.cell_type=="LC4","root_id"].astype("int64").tolist())
LPLC2 = keep("LPLC2", ann.loc[ann.cell_type=="LPLC2","root_id"].astype("int64").tolist())
loom = LC4 + LPLC2
print("loom", len(loom), "GF ok", GF_L in _comp, GF_R in _comp)


DH44 tenuti 6 / 6
CRZ tenuti 6 / 6
HUGIN tenuti 4 / 4
LC4 tenuti 104 / 104
LPLC2 tenuti 210 / 210
loom 314 GF ok True True


In [4]:
def hill(x, K, n=2.0):
    return 0.0 if x<=0 or K<=0 else (x**n)/(K**n + x**n)
NODES = {
 "DH44": dict(ids=DH44, rmax=50.0, wG=1.0, KG=7.0, nG=2.0, wT=0.5, KT=20.0, nT=2.0),
 "CRZ": dict(ids=CRZ, rmax=50.0, wG=1.0, KG=4.0, nG=2.0, wT=0.3, KT=20.0, nT=2.0),
 "HUGIN": dict(ids=HUGIN, rmax=50.0, wG=1.0, KG=7.0, nG=2.0, wT=0.0, KT=1.0, nT=2.0),
}
def drive_hz(node, G, T):
    p = NODES[node]
    return p["rmax"]*(p["wG"]*hill(G,p["KG"],p["nG"])+p["wT"]*hill(T,p["KT"],p["nT"]))
def overlay_ids_and_rate(G, T):
    rates, ids = [], []
    for name,p in NODES.items():
        r = drive_hz(name,G,T); rates.append((name,r,len(p["ids"]))); ids.extend(p["ids"])
    return ids, sum(r for _,r,_ in rates)/len(rates), rates


In [5]:
df_comp = pd.read_csv(config["path_comp"], index_col=0)
df_con = pd.read_parquet(config["path_con"])
fly2i = {int(i):k for k,i in enumerate(df_comp.index.astype("int64"))}
def to_idx(ids):
    return {fly2i[int(i)] for i in ids if int(i) in fly2i}
src = {"DH44": to_idx(DH44), "CRZ": to_idx(CRZ), "HUGIN": to_idx(HUGIN)}
tgt_gf, tgt_loom = to_idx([GF_L,GF_R]), to_idx(loom)
pre, post, w = df_con["Presynaptic_Index"].to_numpy(), df_con["Postsynaptic_Index"].to_numpy(), df_con["Excitatory x Connectivity"].to_numpy()
print("=== 1 HOP ===")
for name,S in src.items():
    mg = np.isin(pre,list(S)) & np.isin(post,list(tgt_gf))
    ml = np.isin(pre,list(S)) & np.isin(post,list(tgt_loom))
    print(f"{name} -> GF n={int(mg.sum())} peso={w[mg].sum():.1f} | looming n={int(ml.sum())} peso={w[ml].sum():.1f}")
mid = set(post[np.isin(pre,list(set().union(*src.values())))].tolist()) - tgt_gf
tg = np.isin(pre,list(mid)) & np.isin(post,list(tgt_gf))
print("=== 2 HOP X->GF ===", int(tg.sum()), float(w[tg].sum()))


=== 1 HOP ===
DH44 -> GF n=0 peso=0.0 | looming n=0 peso=0.0
CRZ -> GF n=0 peso=0.0 | looming n=0 peso=0.0
HUGIN -> GF n=0 peso=0.0 | looming n=0 peso=0.0
=== 2 HOP X->GF === 2 -8.0


In [6]:
def run_trial_phased(exc_loom, exc_state, slnc, path_comp, path_con, params):
    neu, syn, spk_mon = create_model(path_comp, path_con, params)
    poi_state, neu = poi(neu, [], exc_state, params)
    syn = silence(slnc, syn)
    net = Network(neu, syn, spk_mon, *poi_state)
    net.run(duration=params["t_pre"])
    poi_loom, neu = poi(neu, exc_loom, [], params)
    for p in poi_loom:
        net.add(p)
    net.run(duration=params["t_stim"])
    return get_spk_trn(spk_mon)

def run_exp_phased(exp_name, neu_loom, neu_state, params, path_res, path_comp, path_con, force_overwrite=True):
    path_res, path_comp, path_con = map(Path, [path_res, path_comp, path_con])
    path_save = path_res / f"{exp_name}.parquet"
    path_res.mkdir(parents=True, exist_ok=True)
    df_comp = pd.read_csv(path_comp, index_col=0)
    flyid2i = {j:i for i,j in enumerate(df_comp.index)}
    i2flyid = {j:i for i,j in flyid2i.items()}
    missing = [n for n in neu_loom+neu_state if n not in flyid2i]
    if missing:
        raise KeyError(missing[:8])
    exc_loom = [flyid2i[n] for n in neu_loom]
    exc_state = [flyid2i[n] for n in neu_state]
    print(f">>> {exp_name} loom={params['r_poi']} state={params['r_poi2']} n_loom={len(exc_loom)} n_state={len(exc_state)}")
    t0=time()
    res=[run_trial_phased(exc_loom, exc_state, [], str(path_comp), str(path_con), params) for _ in range(int(params["n_run"]))]
    construct_dataframe(res, exp_name, i2flyid).to_parquet(path_save, compression="brotli")
    print(f"    saved {path_save} {time()-t0:.0f}s")
    return path_save


In [7]:
T_FIX=30.0
conditions=[("phased_G2_T30_loom25",2.0,T_FIX),("phased_G20_T30_loom25",20.0,T_FIX)]
for name,G,T in conditions:
    ids2,r2,detail=overlay_ids_and_rate(G,T)
    print(name, "G",G,"T",T,"r_poi2",round(r2,1))
    for n,r,k in detail:
        print(" ",n,round(r,1),"Hz")
    params["r_poi2"]=r2*Hz
    run_exp_phased(name, loom, ids2, params, force_overwrite=True, **config)


phased_G2_T30_loom25 G 2.0 T 30.0 r_poi2 15.1
  DH44 21.1 Hz
  CRZ 20.4 Hz
  HUGIN 3.8 Hz
>>> phased_G2_T30_loom25 loom=25. Hz state=15.07982583 Hz n_loom=314 n_state=16


WARNING    'w' is an internal variable of group 'default_synapses', but also exists in the run namespace with the value array([ 1,  2, 10, ...,  1,  3,  4]). The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


    saved results/gfs_phased/phased_G2_T30_loom25.parquet 176s
phased_G20_T30_loom25 G 20.0 T 30.0 r_poi2 55.0
  DH44 61.9 Hz
  CRZ 58.5 Hz
  HUGIN 44.5 Hz
>>> phased_G20_T30_loom25 loom=25. Hz state=54.95203015 Hz n_loom=314 n_state=16
    saved results/gfs_phased/phased_G20_T30_loom25.parquet 172s


In [8]:
T_PRE_S=float(params["t_pre"]/ms)/1000.0
print("=== INCOLLA QUESTI NUMERI (GFS phased, loom 25 Hz) ===")
for name,G,T in conditions:
    df_spike=utl.load_exps([f"{config['path_res']}/{name}.parquet"])
    df_rate,_=utl.get_rate(df_spike, t_run=params["t_pre"]+params["t_stim"], n_run=params["n_run"])
    print(f"\n{name} G={G} T={T}")
    for lab,nid in [("GF_L",GF_L),("GF_R",GF_R)]:
        sub=df_spike[(df_spike.flywire_id==nid)&(df_spike.t>=T_PRE_S)]
        mu=float(df_rate.loc[nid].iloc[0]) if nid in df_rate.index else 0.0
        if len(sub):
            print(f"  {lab}: rate={mu:.2f} Hz  latenza_post_loom={float(sub.t.min()-T_PRE_S):.4f}s  spikes_post={len(sub)}")
        else:
            print(f"  {lab}: rate={mu:.2f} Hz  NESSUNO SPIKE dopo looming")
    for lab,ids in [("DH44",DH44),("CRZ",CRZ),("HUGIN",HUGIN)]:
        present=[i for i in ids if i in df_rate.index]
        mu=float(df_rate.loc[present].iloc[:,0].mean()) if present else float("nan")
        print(f"  {lab} media {len(present)}: {mu:.2f} Hz")


=== INCOLLA QUESTI NUMERI (GFS phased, loom 25 Hz) ===

phased_G2_T30_loom25 G=2.0 T=30.0
  GF_L: rate=58.00 Hz  latenza_post_loom=0.0204s  spikes_post=58
  GF_R: rate=84.00 Hz  latenza_post_loom=0.0101s  spikes_post=84
  DH44 media 6: 15.17 Hz
  CRZ media 6: 15.83 Hz
  HUGIN media 4: 14.75 Hz

phased_G20_T30_loom25 G=20.0 T=30.0
  GF_L: rate=62.00 Hz  latenza_post_loom=0.0111s  spikes_post=62
  GF_R: rate=77.00 Hz  latenza_post_loom=0.0098s  spikes_post=77
  DH44 media 6: 53.83 Hz
  CRZ media 6: 55.00 Hz
  HUGIN media 4: 52.25 Hz
